In [40]:
import pandas as pd
import spacy
import requests
import os
from spacy.pipeline import EntityRuler
import gc
import numpy as np
from spacy.lang.es.stop_words import STOP_WORDS as STOP_ES
from spacy.lang.en.stop_words import STOP_WORDS as STOP_EN

STOP_WORDS = STOP_ES.union(STOP_EN)

In [41]:
# --- PASO 1: Cargar el Diccionario Tech (Igual que antes) ---
def get_tech_dictionary(limit = 100):
    tags = []
    # La API entrega 100 por página
    for page in range(1, (limit // 100) + 1):
        url = f"https://api.stackexchange.com/2.3/tags?page={page}&pagesize=100&order=desc&sort=popular&site=stackoverflow"
        try:
            response = requests.get(url)
            data = response.json()
            for item in data['items']:
                tags.append(item['name'])
        except Exception as e:
            print(f"Error en página {page}: {e}")
            break
    return tags

In [ ]:
# --- PASO 2: Configurar spaCy ---
print("Cargando modelo y configurando NER...")
nlp = spacy.load("es_core_news_lg")

# 1. Definir excepciones (tecnologías que son stop words pero queremos conservar)
# Usamos minúsculas aquí para la comparación
excepciones_tech = {"go", "r"}

palabras_populares = {
    "performance", "testing", "security", "string", "array", "list", 
    "function", "object", "api", "rest", "json", "documentation",
    "deployment", "automation", "clean", "move", "code", "work",
    "linux", "windows", "ios", "android", "version-control"
}

# 2. Cargar diccionario de GitHub
tech_dictionary = get_tech_dictionary()

diccionario_final = set(tech_dictionary).union(excepciones_tech).difference(palabras_populares)

ruler = nlp.add_pipe("entity_ruler")
LABEL_NAME = "TECH"

# 3. Construir patrones filtrando Stop Words (evita que entre "al", "del", etc.)
patterns = []
for t in diccionario_final:
    t_lower = t.lower()
    
    # Si la palabra es una Stop Word y NO está en nuestras excepciones, la ignoramos
    if t_lower in STOP_WORDS and t_lower not in excepciones_tech:
        continue
    
    # Si es una excepción muy corta (como 'C', 'R', 'AL'), usamos TEXT para evitar minúsculas basura
    if len(t) <= 2:
        patterns.append({"label": LABEL_NAME, "pattern": [{"TEXT": t}]})
    else:
        # Para el resto, detección normal insensible a mayúsculas
        patterns.append({"label": LABEL_NAME, "pattern": [{"LOWER": t_lower}]})

ruler.add_patterns(patterns)
print(f"Diccionario configurado con {len(patterns)} patrones activos.")

# --- PASO 3: Limpieza extrema de RAM ---
if "ner" in nlp.pipe_names: nlp.remove_pipe("ner")
if "parser" in nlp.pipe_names: nlp.remove_pipe("parser")
if "lemmatizer" in nlp.pipe_names: nlp.remove_pipe("lemmatizer")
if "attribute_ruler" in nlp.pipe_names: nlp.remove_pipe("attribute_ruler")

# Vaciamos los vectores (ahorro de ~500MB por cada proceso de n_process)
nlp.vocab.vectors.clear()

print("✅ Modelo optimizado, ligero y sin falsos positivos ('al').")

Cargando modelo y configurando NER...
Diccionario configurado con 98 patrones activos.
✅ Modelo optimizado, ligero y sin falsos positivos ('al').


In [43]:
# --- PASO 3: Cargar tu DataFrame Local ---
# Cambia 'tu_archivo.csv' por el nombre real del archivo que sacaste de la Raspberry
file_path = 'dataset_limpio.csv' 

df = pd.DataFrame()  # Inicializamos un DataFrame vacío por si el archivo no se encuentra
if not os.path.exists(file_path):
    print(f"❌ No se encuentra el archivo {file_path}. Asegúrate de que esté en la misma carpeta.")
else:
    df = pd.read_csv(file_path)

def process_safely(df_column, chunk_size=64):
    results = []
    # Aseguramos que los datos sean strings y manejamos nulos
    data = df_column.astype(str).tolist()
    total = len(data)

    print(f"🚀 Iniciando procesamiento de {total} registros...")

    # nlp.pipe es un generador, procesa por lotes (chunks) automáticamente
    # n_process=-1 activa el multiprocesamiento si tu CPU lo permite
    for i, doc in enumerate(nlp.pipe(data, batch_size=chunk_size, n_process=2)):
        
        # Extraemos las tecnologías detectadas por el EntityRuler (etiqueta 'TEC')
        # Usamos set() para evitar duplicados en el mismo texto (ej: "Python y Python")
        tech_encontradas = list(set([ent.text.lower() for ent in doc.ents if ent.label_ == LABEL_NAME]))
        
        # Si la lista está vacía, guardamos NaN para limpieza posterior
        results.append(tech_encontradas if tech_encontradas else np.nan)
        
        # Imprimir progreso cada vez que terminamos un chunk
        if (i + 1) % chunk_size == 0 or (i + 1) == total:
            print(f"✅ Procesados: {i + 1}/{total}")
            
    return results

# --- EJECUCIÓN ---
# Nota: Pasa solo la columna de texto, no el ID
df['description'] = process_safely(df['description'])

🚀 Iniciando procesamiento de 1834 registros...
✅ Procesados: 64/1834
✅ Procesados: 128/1834
✅ Procesados: 192/1834
✅ Procesados: 256/1834
✅ Procesados: 320/1834
✅ Procesados: 384/1834
✅ Procesados: 448/1834
✅ Procesados: 512/1834
✅ Procesados: 576/1834
✅ Procesados: 640/1834
✅ Procesados: 704/1834
✅ Procesados: 768/1834
✅ Procesados: 832/1834
✅ Procesados: 896/1834
✅ Procesados: 960/1834
✅ Procesados: 1024/1834
✅ Procesados: 1088/1834
✅ Procesados: 1152/1834
✅ Procesados: 1216/1834
✅ Procesados: 1280/1834
✅ Procesados: 1344/1834
✅ Procesados: 1408/1834
✅ Procesados: 1472/1834
✅ Procesados: 1536/1834
✅ Procesados: 1600/1834
✅ Procesados: 1664/1834
✅ Procesados: 1728/1834
✅ Procesados: 1792/1834
✅ Procesados: 1834/1834


In [46]:
df['description']# Contamos cuántos NaN hay para entender cuántos registros no tuvieron tecnologías detectadas

0        [apache, bash, shell, git, scala]
1                     [azure, sql, python]
2                  [android, java, python]
3       [rest, docker, apache, postgresql]
4                            [sql, python]
                       ...                
1829                    [wordpress, linux]
1830                               [excel]
1831                           [html, css]
1832                            [c, excel]
1833                               [azure]
Name: description, Length: 1834, dtype: object

In [45]:
# --- PASO 5: Guardar resultado ---
output_name = "dataset_tratado.csv"
df.to_csv(output_name, index=False)
print(f"✅ ¡Hecho! Archivo guardado como: {output_name}")

✅ ¡Hecho! Archivo guardado como: dataset_tratado.csv
